# Liputan6 Translation (Indonesian -> English)

Produces `translate/{id}.txt` (English) for every article. The AMR parser
needs these because the model is a **concat** model (`id_ID <indo> en_XX <eng>`).

## Before running
1. **Add data source:** `liputan6-data` (must contain `analysis_data.csv`).
2. **Settings -> Accelerator:** GPU **T4**.
3. **Settings -> Internet: ON** (needed to download the NLLB model).
4. Run All.
5. When finished, **Save Version** so the `translate/` output is persisted 
   as this notebook's output (you will add it as a data source to the parser notebook).

It is **resumable**: re-running skips IDs already translated.

In [ ]:
import os, torch, pandas as pd
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

DATA_PATH  = "/kaggle/input/liputan6-data"
CSV_PATH   = os.path.join(DATA_PATH, "analysis_data.csv")
OUTPUT_DIR = "/kaggle/working/translate"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device      :", device)
print("CSV exists  :", os.path.exists(CSV_PATH))

## Load NLLB model & tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/nllb-200-distilled-1.3B"
# src_lang MUST be set so Indonesian is tokenized/flagged correctly
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang="ind_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()
eng_id = tokenizer.convert_tokens_to_ids("eng_Latn")
print("NLLB loaded. eng_Latn id =", eng_id)

## Dataset & DataLoader

In [ ]:
class TextDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path, dtype={"id": str})
        print("Total rows:", len(self.df))
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        return {"id": r["id"], "text": str(r["text"])}

def collate(batch):
    return [b["id"] for b in batch], [b["text"] for b in batch]

ds = TextDataset(CSV_PATH)
# batch_size 4 keeps the 1.3B model within T4 (16GB) memory at max_length=1024
loader = DataLoader(ds, batch_size=4, collate_fn=collate)

## Translate (resumable, bug-fixed: saves the ENGLISH output)

In [ ]:
done, skipped = 0, 0
for ids, texts in tqdm(loader, desc="Translating"):
    keep_ids, keep_texts = [], []
    for i, t in zip(ids, texts):
        if os.path.exists(os.path.join(OUTPUT_DIR, f"{i}.txt")):
            skipped += 1
            continue
        keep_ids.append(i)
        keep_texts.append(t)
    if not keep_ids:
        continue
    enc = tokenizer(keep_texts, return_tensors="pt", padding=True,
                    truncation=True, max_length=1024).to(device)
    with torch.no_grad():
        gen = model.generate(**enc, forced_bos_token_id=eng_id,
                             max_length=1024, num_beams=1)
    outs = tokenizer.batch_decode(gen, skip_special_tokens=True)
    for i, o in zip(keep_ids, outs):
        with open(os.path.join(OUTPUT_DIR, f"{i}.txt"), "w", encoding="utf-8") as f:
            f.write(o)          # <-- saves the TRANSLATION (original bug saved the Indonesian)
    done += len(keep_ids)

n_files = len([f for f in os.listdir(OUTPUT_DIR) if f.endswith('.txt')])
print(f"Newly translated: {done} | already-done skipped: {skipped}")
print(f"Total translation files in {OUTPUT_DIR}: {n_files}")
print("\n>>> Now click 'Save Version' so this translate/ output is persisted.")